# GuacaMol Dataset Tutorial

[GuacaMol](https://github.com/BenevolentAI/guacamol) (Brown et al., 2019) is a benchmark dataset
derived from ChEMBL containing ~1.6 million drug-like SMILES strings. Each molecule has 10 RDKit
physicochemical properties: `BertzCT`, `MolLogP`, `MolWt`, `TPSA`, `NumHAcceptors`, `NumHDonors`,
`NumRotatableBonds`, `NumAliphaticRings`, `NumAromaticRings`, and `QED`.

This tutorial demonstrates the `GuacaMol` dataset class from `alf_tools`:
1. Downloading and inspecting the raw SMILES files
2. Loading the dataset and building a pandas DataFrame
3. Visualising property distributions and correlations
4. Querying properties for arbitrary molecules
5. Case study: aspirin's physicochemical profile

We use `max_molecules=10_000` throughout so the notebook runs in under a minute on CPU.

## Section 0 — Environment Setup

Run `uv sync` from the `tutorials/` directory to install all dependencies, then select the `.venv`
kernel when prompted.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    ["uv", "pip", "install", "--python", sys.executable, "-e", ".."],
)
print("✓ Environment ready")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from alf_core.dataclasses.candidate import Modality
from alf_tools.datasets.guacamol import (
    ALL_PROPERTIES,
    GUACAMOL_FILES,
    GuacaMol,
    GuacaMolConfig,
    download_guacamol,
)

DATA_DIR = Path.home() / ".cache" / "alf"
PROPERTY_COLS = sorted(ALL_PROPERTIES)

print("✓ Imports OK")
print(f"Properties ({len(PROPERTY_COLS)}): {PROPERTY_COLS}")

## Section 1 — Download

`download_guacamol` streams all four GuacaMol SMILES files from Figshare and caches them locally.
With `max_lines=10_000`, only the first 10 000 lines of each file are written to disk; subsequent
runs detect the existing files and skip the download entirely.

In [ ]:
download_guacamol(data_dir=DATA_DIR, max_lines=10_000)

print("Downloaded files:")
for split, info in GUACAMOL_FILES.items():
    filepath = DATA_DIR / info["name"]
    size_kb = filepath.stat().st_size / 1024
    print(f"  {split:5s}  {info['name']}  ({size_kb:.1f} KB)")

In [ ]:
all_smiles_path = DATA_DIR / GUACAMOL_FILES["ALL"]["name"]
with open(all_smiles_path, encoding="utf-8") as f:
    sample = [next(f).strip() for _ in range(5)]

print("First 5 SMILES from the corpus:")
for i, smi in enumerate(sample, 1):
    print(f"  {i}. {smi}")

## Section 2 — Load & Build DataFrame

We instantiate `GuacaMolConfig` with `target_property="QED"` and `max_molecules=10_000`, then
`GuacaMol` loads the corpus and computes all 10 RDKit properties for each molecule.

Each `Candidate` carries its SMILES string in `.data` and a dict of all 10 computed properties
in `.features`. We flatten these into a pandas DataFrame for analysis.

In [ ]:
config = GuacaMolConfig(
    name="guacamol_tutorial",
    modality=Modality.SEQUENCE,
    seed=42,
    train_ratio=0.8,
    validation_frac=0.1,
    test_ratio=0.1,
    target_property="QED",
    max_molecules=10_000,
    data_dir=DATA_DIR,
)
dataset = GuacaMol(config)

n = len(dataset._raw_dataset.candidates)
print(dataset)
print(f"Loaded {n} candidates")

In [ ]:
rows = []
for candidate in dataset._raw_dataset.candidates:
    row = {"smiles": candidate.data}
    row.update({prop: candidate.features[prop] for prop in PROPERTY_COLS})
    rows.append(row)

df = pd.DataFrame(rows)[["smiles"] + PROPERTY_COLS]
print(f"Shape: {df.shape}")
df.describe()

## Section 3 — Dataset Statistics

We examine the distribution of SMILES string lengths, all 10 physicochemical properties
individually, and their pairwise correlations.

In [ ]:
df["smiles_len"] = df["smiles"].str.len()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["smiles_len"], kde=True, ax=ax)
ax.set_xlabel("SMILES string length (characters)")
ax.set_title("Distribution of SMILES Lengths (n=10 000)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, prop in zip(axes.flat, PROPERTY_COLS):
    sns.histplot(df[prop], kde=True, ax=ax)
    ax.set_title(prop, fontsize=11)
    ax.set_xlabel("")
fig.suptitle("Distribution of RDKit Physicochemical Properties (n=10 000)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[PROPERTY_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Property Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

## Section 4 — Querying

`dataset.query()` accepts a list of `Candidate` objects and returns a `LabelledCandidates` with
the `target_property` label for each molecule. For SMILES already in the corpus the stored label
is returned directly; for novel molecules RDKit computes it on the fly.

**Note:** `query()` returns only the `target_property` label (`QED` here) — not all 10 properties.
See Section 5 for how to retrieve the full property profile for a single molecule.

In [ ]:
QUERY_MOLECULES = {
    "aspirin":   "CC(=O)Oc1ccccc1C(=O)O",
    "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "caffeine":  "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
}

query_candidates = [
    Candidate(data=smi, modality=Modality.SEQUENCE)
    for smi in QUERY_MOLECULES.values()
]

results = dataset.query(query_candidates)

print(f"Return type : {type(results).__name__}")
print(f"Labels shape: {results.labels.shape}")
print(f"\nFirst result:")
print(f"  SMILES : {results.candidates[0].data}")
print(f"  QED    : {results.labels[0]:.4f}")

In [ ]:
query_rows = [
    {"molecule": name, "smiles": cand.data, "QED (label)": float(label)}
    for name, cand, label in zip(
        QUERY_MOLECULES.keys(), results.candidates, results.labels
    )
]
pd.DataFrame(query_rows).set_index("molecule")